# Cross-experiment APA hit overlap: treatment vs pseudotime

Compares the gene-level hits from:
- Treatment experiment (`UTR_treatment/`): genes that change APA under CRISPR loss of function of 36 ASD-risk genes, holding pseudotime fixed. DRIMSeq pseudobulk main-effect LRT.
- Pseudotime experiment (`UTR/`): genes that change APA over the neurogenesis trajectory in control (uninduced) cells. Single-cell Dirichlet-multinomial regression with spline of pseudotime.

Both come from the same dataset. We test whether the same genes are APA-responsive in both contexts and whether the directions agree.

## What this notebook tests

1. Hypergeometric overlap at gene level (per lineage), with Fisher's exact test.
2. Direction of effect agreement for shared genes: whether treatment-lengthening genes also lengthen with pseudotime.
3. Quantitative correlation of Δψ values between the two experiments (Spearman + Pearson) for the shared gene set.
4. Doubly-cross-validated genes: genes overlapping in both lineages, i.e. the most context-independent APA-plastic genes.

## Inputs

- `dirmult_treatment_results/<lineage>/composite_ranking/composite_ranking_<lineage>.csv` (treatment)
- `composite_ranking/composite_ranking_optimized2_<lineage>.csv` (pseudotime; ENSEMBL keyed, with `gene_name` = symbol column for joining)

## Outputs

- `dirmult_treatment_results/cross_experiment_overlap/` directory with:
  - `overlap_gene_table_<lineage>.csv`: per-gene cross-tabulation
  - `overlap_doubly_validated.csv`: genes overlapping in both lineages
  - `overlap_summary.csv`: per-lineage Fisher / Spearman summary
  - Figures: Venn, UpSet, scatter plots, direction-agreement bar chart


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR_treatment")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from scipy.stats import fisher_exact, spearmanr, pearsonr, binomtest

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 7,
    "font.family": "sans-serif", "font.sans-serif": ["Arial","Helvetica","DejaVu Sans"],
    "axes.titlesize": 8, "axes.labelsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.8,
})
print("Setup complete.")

In [ ]:
LINEAGES = ["inhibitory", "excitatory"]
PADJ_THRESHOLD = 0.05

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"

def treat_comp_csv(ln): return UTR_DIR / "dirmult_treatment_results" / ln / "composite_ranking" / f"composite_ranking_{ln}.csv"
def pseudo_comp_csv(ln): return UTR_DIR / "composite_ranking" / f"composite_ranking_optimized2_{ln}.csv"

out_dir = UTR_DIR / "dirmult_treatment_results" / "cross_experiment_overlap"
fig_dir = out_dir / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

for ln in LINEAGES:
    for p in [treat_comp_csv(ln), pseudo_comp_csv(ln)]:
        assert p.exists(), f"missing: {p}"
    print(f"  {ln:12s}  inputs present")
print(f"Output:  {out_dir}")

## Step 1: load both experiments' hit tables per lineage and join on gene symbol

In [ ]:
def load_pair(lineage):
    # Treatment: gene_id is the HGNC symbol; padj_treatment is the LRT padj
    treat = pd.read_csv(treat_comp_csv(lineage))
    treat = treat.rename(columns={"gene_id": "symbol"})
    treat["sig_treat"] = treat["padj_treatment"] < PADJ_THRESHOLD
    # Pseudotime: gene_id = ENSEMBL; gene_name = HGNC symbol; padj is the LRT padj
    pseudo = pd.read_csv(pseudo_comp_csv(lineage))
    pseudo = pseudo.rename(columns={"gene_name": "symbol"})
    pseudo["sig_pseudo"] = pseudo["padj"] < PADJ_THRESHOLD
    # Common universe = intersection of symbol-keys
    common = set(treat["symbol"].dropna()) & set(pseudo["symbol"].dropna())
    # Join only by symbol (one-to-one within each table after dedupe)
    treat_u = treat.drop_duplicates("symbol")
    pseudo_u = pseudo.drop_duplicates("symbol")
    df = treat_u[["symbol","sig_treat","padj_treatment","delta_psi_treatment",
                  "pvalue_treatment","n_features_post_filter"]].merge(
         pseudo_u[["symbol","sig_pseudo","padj","delta_psi","rho_W","K",
                   "trajectory_pattern","p_value_chisq"]],
         on="symbol", how="inner")
    df["sig_both"] = df["sig_treat"] & df["sig_pseudo"]
    df["lineage"] = lineage
    return df, len(treat), len(pseudo), len(common)

pairs = {}
for ln in LINEAGES:
    df, n_t, n_p, n_c = load_pair(ln)
    pairs[ln] = df
    n_sig_t = int(df["sig_treat"].sum()); n_sig_p = int(df["sig_pseudo"].sum())
    print(f"\n[{ln}]")
    print(f"  treatment input:    {n_t:,}  pseudotime input: {n_p:,}  common universe: {n_c:,}")
    print(f"  joined rows:        {len(df):,}")
    print(f"  treatment hits in common: {n_sig_t}")
    print(f"  pseudotime hits in common: {n_sig_p}")
    print(f"  overlap (both sig):       {int(df['sig_both'].sum())}")


## Step 2: hypergeometric overlap test per lineage

In [ ]:
def fisher_overlap(df, hit_col_treat, hit_col_pseudo):
    a = int(df[df[hit_col_treat] & df[hit_col_pseudo]].shape[0])
    b = int(df[df[hit_col_treat] & ~df[hit_col_pseudo]].shape[0])
    c = int(df[~df[hit_col_treat] & df[hit_col_pseudo]].shape[0])
    d = int(df[~df[hit_col_treat] & ~df[hit_col_pseudo]].shape[0])
    if (a+b) == 0 or (a+c) == 0:
        return dict(a=a, b=b, c=c, d=d, fold=np.nan, fisher_p=np.nan,
                     expected=np.nan, n_a=a+b, n_b=a+c, universe=a+b+c+d)
    expected = (a+b) * (a+c) / (a+b+c+d)
    fold = a / expected if expected > 0 else np.nan
    _, p = fisher_exact([[a, b], [c, d]], alternative="greater")
    return dict(a=a, b=b, c=c, d=d, fold=fold, expected=expected, fisher_p=p,
                 n_treat=a+b, n_pseudo=a+c, universe=a+b+c+d)

rows = []
for ln in LINEAGES:
    df = pairs[ln]
    disc = fisher_overlap(df, "sig_treat", "sig_pseudo")
    rows.append({"lineage": ln, "tier": "discovery",
                  "n_treat": disc["n_treat"], "n_pseudo": disc["n_pseudo"],
                  "universe": disc["universe"], "overlap": disc["a"],
                  "expected": disc["expected"], "fold": disc["fold"],
                  "fisher_p_one_sided": disc["fisher_p"]})
summary_df = pd.DataFrame(rows)
summary_df.to_csv(out_dir / "overlap_summary.csv", index=False)
print("\n=== Per-lineage / per-tier Fisher overlap ===")
with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"):
    print(summary_df.to_string(index=False))

## Step 3: gene-level cross-tabulations + write per-lineage tables

In [ ]:
for ln in LINEAGES:
    df = pairs[ln]
    out_path = out_dir / f"overlap_gene_table_{ln}.csv"
    df_out = df[["symbol","lineage","sig_treat","sig_pseudo",
                  "padj_treatment","padj","pvalue_treatment","p_value_chisq",
                  "delta_psi_treatment","delta_psi","rho_W","n_features_post_filter","K",
                  "trajectory_pattern"]].copy()
    df_out.to_csv(out_path, index=False)
    print(f"[{ln}] wrote {out_path}  shape={df_out.shape}")

# Doubly-cross-validated: genes overlapping in both lineages
overlap_per_ln = {ln: set(pairs[ln].query("sig_treat & sig_pseudo")["symbol"]) for ln in LINEAGES}
doubly = overlap_per_ln["inhibitory"] & overlap_per_ln["excitatory"]

print(f"\nDoubly-cross-validated (overlap in both lineages, discovery tier): {len(doubly)}")
print(f"  genes: {sorted(doubly)}")

# Build the doubly-validated CSV with per-lineage details
rows = []
for sym in sorted(doubly):
    row = {"symbol": sym}
    for ln in LINEAGES:
        sub = pairs[ln].set_index("symbol").loc[sym]
        row[f"{ln}_padj_treat"] = sub["padj_treatment"]
        row[f"{ln}_padj_pseudo"] = sub["padj"]
        row[f"{ln}_dpsi_treat"] = sub["delta_psi_treatment"]
        row[f"{ln}_dpsi_pseudo"] = sub["delta_psi"]
    rows.append(row)
doubly_df = pd.DataFrame(rows)
if len(doubly_df):
    doubly_df.to_csv(out_dir / "overlap_doubly_validated.csv", index=False)
    print(f"\nWrote doubly_validated CSV ({len(doubly_df)} rows)")
    print(doubly_df.to_string(index=False))

## Step 4: direction-of-effect agreement (sign concordance + correlation)

In [ ]:
direction_rows = []
for ln in LINEAGES:
    df = pairs[ln]
    ov = df[df["sig_both"] & df["delta_psi_treatment"].notna() & df["delta_psi"].notna()]
    if len(ov) >= 3:
        same_sign = int((np.sign(ov["delta_psi_treatment"]) == np.sign(ov["delta_psi"])).sum())
        n = len(ov)
        bin_p = binomtest(same_sign, n, 0.5, alternative="greater").pvalue
        rho_s, p_s = spearmanr(ov["delta_psi_treatment"], ov["delta_psi"])
        rho_p, p_p = pearsonr(ov["delta_psi_treatment"], ov["delta_psi"])
    else:
        same_sign = n = 0; bin_p = rho_s = p_s = rho_p = p_p = np.nan
    direction_rows.append(dict(
        lineage=ln, n_overlap_with_dpsi=n, same_sign=same_sign,
        sign_concordance=same_sign / n if n else np.nan,
        binom_p_concord_gt_0p5=bin_p,
        spearman_rho=rho_s, spearman_p=p_s,
        pearson_rho=rho_p, pearson_p=p_p,
    ))
direction_df = pd.DataFrame(direction_rows)
direction_df.to_csv(out_dir / "overlap_direction_agreement.csv", index=False)
print("\n=== Direction of effect agreement on overlap ===")
with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"):
    print(direction_df.to_string(index=False))

## Step 5: Venn diagrams (one per lineage, discovery tier)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, ln in zip(axes, LINEAGES):
    df = pairs[ln]
    a = int(df.query("sig_treat & ~sig_pseudo").shape[0])
    b = int(df.query("~sig_treat & sig_pseudo").shape[0])
    ab = int(df.query("sig_treat & sig_pseudo").shape[0])
    venn2(subsets=(a, b, ab), set_labels=("treatment", "pseudotime"), ax=ax)
    summary = summary_df.query("lineage == @ln & tier == 'discovery'").iloc[0]
    ax.set_title(f"{ln} (discovery tier)\n"
                  f"fold={summary['fold']:.2f}x, Fisher p={summary['fisher_p_one_sided']:.2e}")
fig.savefig(fig_dir / "venn_discovery.pdf", bbox_inches="tight", dpi=300)
fig.savefig(fig_dir / "venn_discovery.png", bbox_inches="tight", dpi=200)
plt.show()

## Step 6: scatter of Δψ_treatment vs Δψ_pseudotime for shared genes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, ln in zip(axes, LINEAGES):
    df = pairs[ln]
    # All genes with both dpsi values (not just hits)
    valid = df[["delta_psi_treatment","delta_psi"]].notna().all(axis=1)
    base = df[valid]
    # Only-overlap (both sig)
    ov = df.query("sig_both & delta_psi_treatment.notna() & delta_psi.notna()")
    ax.scatter(base["delta_psi"], base["delta_psi_treatment"],
                s=2, alpha=0.15, c="#bbbbbb", rasterized=True, label=f"all (n={base.shape[0]:,})")
    ax.scatter(ov["delta_psi"], ov["delta_psi_treatment"],
                s=22, alpha=0.85, c="#e74c3c", rasterized=True,
                edgecolors="black", linewidth=0.2,
                label=f"both significant (n={ov.shape[0]})")
    # Annotate doubly-cross-validated genes
    for sym in doubly:
        if sym in ov["symbol"].values:
            r = ov.set_index("symbol").loc[sym]
            ax.annotate(sym, (r["delta_psi"], r["delta_psi_treatment"]),
                         xytext=(4, 4), textcoords="offset points", fontsize=6,
                         fontweight="bold")
    if len(ov) >= 3:
        rho, _ = spearmanr(ov["delta_psi"], ov["delta_psi_treatment"])
    else:
        rho = np.nan
    ax.axhline(0, color="grey", lw=0.5, ls=":")
    ax.axvline(0, color="grey", lw=0.5, ls=":")
    lim = max(abs(base["delta_psi"]).max(), abs(base["delta_psi_treatment"]).max(), 0.5) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "--", color="black", lw=0.4, alpha=0.5, label="x = y")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel("Δψ over pseudotime (uninduced cells)")
    ax.set_ylabel("Δψ treatment (induced - uninduced)")
    ax.set_title(f"{ln}  Spearman on overlap = {rho:.3f}" if not np.isnan(rho) else ln)
    ax.legend(fontsize=6, frameon=False, loc="lower right")
fig.savefig(fig_dir / "scatter_dpsi_treat_vs_pseudo.pdf", bbox_inches="tight", dpi=300)
fig.savefig(fig_dir / "scatter_dpsi_treat_vs_pseudo.png", bbox_inches="tight", dpi=200)
plt.show()

## Step 7: sign-concordance bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, ln in zip(axes, LINEAGES):
    df = pairs[ln]
    ov = df[df["sig_both"] & df["delta_psi_treatment"].notna() & df["delta_psi"].notna()]
    if len(ov) == 0:
        ax.text(0.5, 0.5, "no overlap genes with valid Δψ", ha="center", va="center")
        ax.set_title(ln); continue
    same = int((np.sign(ov["delta_psi_treatment"]) == np.sign(ov["delta_psi"])).sum())
    diff = len(ov) - same
    ax.bar(["same sign", "different sign"], [same, diff],
            color=["#16a085","#c0392b"], edgecolor="black", linewidth=0.4)
    for i, v in enumerate([same, diff]):
        ax.text(i, v + 0.1, str(v), ha="center", va="bottom", fontsize=8)
    ax.axhline(len(ov) / 2, color="black", lw=0.6, ls="--",
                label=f"50% (random)")
    ax.set_ylabel(f"# overlap genes")
    direction = direction_df.query("lineage == @ln").iloc[0]
    binp = direction['binom_p_concord_gt_0p5']
    binp_str = f"binomial p={binp:.3g}" if pd.notna(binp) else "binomial p=NA"
    ax.set_title(f"{ln}: {same}/{len(ov)} same-sign\n{binp_str}")
    ax.legend(fontsize=6, frameon=False, loc="upper right")
fig.savefig(fig_dir / "sign_concordance_bar.pdf", bbox_inches="tight", dpi=300)
fig.savefig(fig_dir / "sign_concordance_bar.png", bbox_inches="tight", dpi=200)
plt.show()

## Step 8: threshold sensitivity of the overlap (padj threshold of the treatment hits)

In [ ]:
# Sensitivity: vary the padj threshold for treatment and see how the overlap scales
thresholds = [0.001, 0.005, 0.01, 0.05, 0.10]
rows = []
for ln in LINEAGES:
    df = pairs[ln]
    pseudo_sig = df["padj"] < PADJ_THRESHOLD  # fix pseudo at 0.05
    n_pseudo = int(pseudo_sig.sum())
    for thr in thresholds:
        treat_sig = df["padj_treatment"] < thr
        n_treat = int(treat_sig.sum())
        n_overlap = int((treat_sig & pseudo_sig).sum())
        univ = len(df)
        exp = n_treat * n_pseudo / univ if univ else np.nan
        fold = n_overlap / exp if exp > 0 else np.nan
        _, p = fisher_exact([[n_overlap, n_treat - n_overlap],
                              [n_pseudo - n_overlap, univ - n_treat - n_pseudo + n_overlap]],
                              alternative="greater") if univ else (np.nan, np.nan)
        rows.append(dict(lineage=ln, treat_padj_threshold=thr, n_treat=n_treat,
                          n_pseudo=n_pseudo, n_overlap=n_overlap, expected=exp,
                          fold=fold, fisher_p=p))
sens_df = pd.DataFrame(rows)
sens_df.to_csv(out_dir / "overlap_sensitivity.csv", index=False)
print("\n=== Sensitivity: overlap fold + Fisher across treatment padj thresholds (pseudo padj fixed at 0.05) ===")
with pd.option_context("display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"):
    print(sens_df.to_string(index=False))

## Step 9: ASD-relevance of overlap genes

In [ ]:
SFARI_CSV = Path(f"{DATA_ROOT}/common_files/annotations/SFARI-Gene_genes_03-22-2026release_03-27-2026export.csv")
sfari = pd.read_csv(SFARI_CSV)
SFARI_SYMBOLS = set(sfari["gene-symbol"].dropna())
sfari_score = sfari.set_index("gene-symbol")["gene-score"].to_dict()

import anndata
a_back = anndata.read_h5ad(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad", backed="r")
CHOOSE = set(g for g in a_back.obs["gRNA"].dropna().astype(str).unique() if g != "Control2")

for ln in LINEAGES:
    df = pairs[ln]
    overlap_genes = df.query("sig_both")[["symbol","padj_treatment","padj",
                                            "delta_psi_treatment","delta_psi"]].copy()
    overlap_genes["is_sfari"] = overlap_genes["symbol"].isin(SFARI_SYMBOLS)
    overlap_genes["sfari_score"] = overlap_genes["symbol"].map(sfari_score)
    overlap_genes["is_choose_target"] = overlap_genes["symbol"].isin(CHOOSE)
    print(f"\n=== {ln} overlap genes annotated with ASD context ===")
    print(overlap_genes.sort_values("padj_treatment").to_string(index=False))
    overlap_genes.to_csv(out_dir / f"overlap_asd_annotated_{ln}.csv", index=False)

## Step 10: paragraph summary

In [ ]:
bar = "-" * 70
print(f"\n{bar}\nCROSS-EXPERIMENT OVERLAP SUMMARY\n{bar}\n")
for ln in LINEAGES:
    sd = summary_df.query("lineage == @ln & tier == 'discovery'").iloc[0]
    dd = direction_df.query("lineage == @ln").iloc[0]
    n_doub = len(doubly)
    print(f"\n[{ln}]")
    print(f"  Discovery overlap: {sd['overlap']}/{sd['n_treat']} treatment hits also pseudotime; "
          f"fold={sd['fold']:.2f}x; Fisher p={sd['fisher_p_one_sided']:.2e}")
    if pd.notna(dd['sign_concordance']):
        print(f"  Sign concordance:  {dd['same_sign']}/{dd['n_overlap_with_dpsi']} "
              f"= {dd['sign_concordance']:.0%}; "
              f"binomial p (>50%) = {dd['binom_p_concord_gt_0p5']:.2g}")
        print(f"  Spearman(Δψ_treat, Δψ_pseudo) on overlap: rho={dd['spearman_rho']:.3f} p={dd['spearman_p']:.2g}")
print(f"\nDoubly-cross-validated genes (overlap in both lineages): {sorted(doubly)}")
print(f"\nInterpretation:")
print("  - Significant gene-level overlap (~3x fold over chance) in BOTH lineages")
print("    -> there IS a shared 'APA-plastic' gene set affected by both contexts.")
print("  - Direction of effect (sign of delta_psi) does NOT systematically agree")
print("    -> treatment is not simply 'more pseudotime' nor 'less pseudotime'.")
print("  - Doubly-cross-validated set (overlap in BOTH lineages) identifies the")
print("    most context-independent APA-plastic genes: HSP90AA1, TIMM8B, CD24 etc.")
print("    These are top candidates for orthogonal validation (3'RACE, etc.).")

## Step 11: doubly-validated overlap significance (dependency-aware permutation)

In [ ]:
# Step 11: significance of the doubly-cross-validated (both-lineages) overlap
# Per-lineage overlaps are Fisher-tested above. The 3-gene intersection across both lineages needs a
# null that preserves each experiment's cross-lineage structure (development hits recur across
# lineages; so do perturbation hits) and breaks only the development-perturbation link: permute the
# treatment (EXC,INH) status-pairs across genes against the fixed pseudotime pairs.
from scipy.stats import hypergeom
_rng = np.random.default_rng(0)
def _hs(p, s, pa):
    d = pd.read_csv(p); return set(d.loc[d[pa] < 0.05, s].dropna().astype(str)), set(d[s].dropna().astype(str))
_S, _V = {}, {}
for ln in LINEAGES:
    _S[("t", ln)], _V[("t", ln)] = _hs(treat_comp_csv(ln), "gene_id", "padj_treatment")
    _S[("p", ln)], _V[("p", ln)] = _hs(pseudo_comp_csv(ln), "gene_name", "padj")
_Ua = set.intersection(*_V.values()); _g = sorted(_Ua); _n = len(_g)
_pE, _pI, _tE, _tI = [np.array([x in (_S[k] & _Ua) for x in _g]) for k in
                      [("p","excitatory"),("p","inhibitory"),("t","excitatory"),("t","inhibitory")]]
_obs = int(((_pE & _pI) & (_tE & _tI)).sum()); _B = 20000
_ge = sum(int(((_pE & _pI) & (_tE[q] & _tI[q])).sum()) >= _obs for q in (_rng.permutation(_n) for _ in range(_B)))
_p_perm = (_ge + 1) / (_B + 1)
_p_naive = float(hypergeom.sf(_obs - 1, _n, int((_pE & _tE).sum()), int((_pI & _tI).sum())))
print(f"Doubly-validated overlap: universe={_n}, observed={_obs} genes; "
      f"dependency-aware permutation p={_p_perm:.4f} (B={_B}); naive hypergeometric p={_p_naive:.2e} (anti-conservative)")
pd.DataFrame([{"set":"doubly_cross_validated","n_genes":_obs,"universe":_n,"perm_p_dependency_aware":_p_perm,
               "n_perm":_B,"hypergeom_p_naive":_p_naive}]).to_csv(out_dir/"overlap_doubly_validated_significance.csv", index=False)
